# test_density_v35.py

**Converted from:** `/Users/songkarn/locarb/biomass/handheld-lidar-slam-toolbox/yolov11/test_density_v35.py`

**Description:** This notebook was automatically generated from the Python script for easier execution and modification on another machine.

## Imports

In [ ]:
#!/usr/bin/env python3

## Main Code

In [4]:
"""
Test trained YOLOv11 model (density_v35) on test dataset
Show results for individual images and overall metrics
"""
from ultralytics import YOLO
from pathlib import Path
import shutil

# Load the best trained model from density_v35
model_path = "/home/pun/Desktop/yolov11/runs/detect/density_v5_lightaug/weights/best.pt"
model = YOLO(model_path)

print("=" * 70)
print("Testing YOLOv11 Density Model (v35)")
print("=" * 70)
print(f"Model: {model_path}")

# Path to dataset configuration
data_yaml = '/home/pun/Desktop/yolov11/dataset/data_density.yaml'

# Run validation on test set
print("\n" + "=" * 70)
print("1. Running validation on test dataset...")
print("=" * 70)

try:
    results = model.val(
        data=data_yaml,
        imgsz=320,
        batch=1,
        save_json=True,
        save_hybrid=True,
        plots=True,
        split='test'  # Use test split
    )
    
    print("\n" + "=" * 70)
    print("Validation Metrics:")
    print("=" * 70)
    print(f"Precision: {results.box.p.mean():.3f}")
    print(f"Recall: {results.box.r.mean():.3f}")
    print(f"mAP50: {results.box.map50:.3f}")
    print(f"mAP50-95: {results.box.map:.3f}")
except Exception as e:
    print(f"Validation error: {e}")
    print("Continuing with predictions...")

# Run prediction on test images
print("\n" + "=" * 70)
print("2. Running predictions on test images...")
print("=" * 70)

test_images_dir = Path("yolov11/dataset/images/test/density")
all_test_images = sorted(list(test_images_dir.glob("*.jpg")))
test_images = all_test_images[:10]  # First 10 images

output_dir = Path("test_results_v35")
output_dir.mkdir(exist_ok=True)

print(f"\nFound {len(all_test_images)} test images total")
print(f"Processing first {len(test_images)} images...\n")

total_detections = 0

if len(test_images) == 0:
    print("⚠️  No test images found! Skipping predictions.")
else:
    for i, img_path in enumerate(test_images, 1):
        print(f"[{i}/{len(test_images)}] Processing: {img_path.name}")
        
        # Run prediction
        pred_results = model.predict(
            source=str(img_path),
            imgsz=320,
            conf=0.25,
            save=True,
            save_txt=True,
            save_conf=True,
            project=str(output_dir),
            name='predictions',
            exist_ok=True
        )
        
        # Print detection info
        for r in pred_results:
            boxes = r.boxes
            num_detections = len(boxes)
            total_detections += num_detections
            print(f"  ✓ Detected {num_detections} trees", end="")
            if num_detections > 0:
                print(f" | Confidence: {boxes.conf.min():.3f} - {boxes.conf.max():.3f}")
            else:
                print()

print("\n" + "=" * 70)
print("Results Summary:")
print("=" * 70)
print(f"✓ Processed images: {len(test_images)}")
print(f"✓ Total detections: {total_detections}")
if len(test_images) > 0:
    print(f"✓ Average detections per image: {total_detections/len(test_images):.1f}")
print(f"✓ Predictions saved to: {output_dir / 'predictions'}")
print("=" * 70)

# Copy validation plots from training run
val_dir = Path("/home/pun/Desktop/notebooks_density/training/runs/detect/density_training3")
files_to_copy = [
    ("confusion_matrix.png", "confusion_matrix.png"),
    ("results.png", "training_metrics.png"),
    ("val_batch0_pred.jpg", "validation_predictions.jpg"),
    ("BoxPR_curve.png", "precision_recall_curve.png")
]

print("\nCopying training/validation plots...")
for src_name, dst_name in files_to_copy:
    src_path = val_dir / src_name
    if src_path.exists():
        shutil.copy(src_path, output_dir / dst_name)
        print(f"✓ Copied: {dst_name}")

print("\n" + "=" * 70)
print("🎉 Testing Complete!")
print("=" * 70)
print(f"\nView results: open {output_dir}")
print(f"Predictions: open {output_dir / 'predictions'}")
print("=" * 70)


Testing YOLOv11 Density Model (v35)
Model: /home/pun/Desktop/yolov11/runs/detect/density_v5_lightaug/weights/best.pt

1. Running validation on test dataset...
WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.4.21 🚀 Python-3.11.15 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)


YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 884.4±291.8 MB/s, size: 16.9 KB)
val: Scanning /home/pun/Desktop/notebooks_density/dataset_creation/yolov11/dataset/labels/test... 3 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3/3 376.3it/s 0.0s
val: New cache created: /home/pun/Desktop/notebooks_density/dataset_creation/yolov11/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.9it/s 1.0s5.9s
                   all          3        228      0.975      0.521      0.899      0.492
Speed: 1.7ms preprocess, 57.0ms inference, 0.0ms loss, 1.6ms postprocess per image
Saving /home/pun/Desktop/notebooks_density/testing/runs/detect/val4/predictions.json...
Results saved to /home/pun/Desktop/notebooks_density/testing/runs/detect/val4

Validation Metrics:
Precision: 0.975
Recall: 0.521
mAP50: 0.899
mAP50-95: 0.492

2. 